# A2.2 · Bootstrapping the first credential

**Function A — Securing AI Architectures → Securing the Architecture — Identity and Ingress**  ·  *Security of AI*

Builds on **[A2.1 · Agent identity: user, workload, agent](https://spbreed.github.io/cyber-commons/lessons/A2.1.html)**.

| | |
|---|---|
| Open-source tooling | SPIFFE/SPIRE |
| Open-weight models | — |
| Frontier models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

An agent needs a credential to prove who it is, and it cannot be given one safely without already proving who it is. Every long-lived secret in your estate exists because somebody resolved that circle by giving up.

> **At CyberTravels.** Each of TripBot's four agents needs a credential to prove it is that agent, and cannot be handed one safely without already proving it. The long-lived bearer token in R5 exists because somebody resolved that circle by giving up.

## 2 · The framework

```
   to get a credential you must prove who you are
   to prove who you are you need a credential
                  |
             attestation breaks the circle
                  v
   platform says "this workload is what it claims"  (hardware/orchestrator)
                  |
                  v
   short-lived identity document ---> rotated automatically, never stored
```

**Mitigates: T9 Identity Spoofing & Impersonation.**

A2.1 says the workload needs its own identity. This lesson is about how it gets
one, because there is a circularity: to receive a credential securely the
workload must already prove who it is.

The wrong answer is a **pre-shared secret** — a key in the image, a token in an
environment variable, a file mounted at deploy time. All of them are copyable,
and a copyable secret makes possession the proof of identity. Anyone who reads
the image is the agent.

The control is **attestation**. The platform that started the workload already
knows things nobody else can forge: which image ran, in which namespace, under
which service account, on which node. It signs a statement to that effect, and
an identity service exchanges that statement for a short-lived credential bound
to that workload.

Three properties matter:

- **Non-copyable.** The attestation describes a running process. Copying the
  document to another machine produces a claim the platform will not sign.
- **Short-lived.** The credential expires in minutes, so theft has a deadline.
- **Bound.** It is issued *to* that workload identity, so presenting it from
  elsewhere fails.

This is what SPIFFE/SPIRE and every cloud workload-identity system do. The
lesson models the exchange, not the product.

> **What this control closes.**
>
> Makes **possession stop being proof**. Without it, A1.7 is unavoidable: a copyable secret means every holder is the agent.

## 3 · The control

In [ ]:
import hashlib, time

PLATFORM_TRUTH = {          # only the platform can observe these
 "proc-1": {"image": "reports-agent@sha256:aa11", "namespace": "prod", "node": "n-7"},
 "proc-2": {"image": "billing-agent@sha256:bb22", "namespace": "prod", "node": "n-7"},
}

def platform_attest(pid):
    """The platform signs a statement about a process it actually started."""
    facts = PLATFORM_TRUTH.get(pid)
    if not facts:
        return None                       # cannot attest a process it did not start
    payload = f"{pid}|{facts['image']}|{facts['namespace']}"
    return {"claims": facts, "sig": hashlib.sha256(payload.encode()).hexdigest()[:16]}

REGISTERED = {"reports-agent@sha256:aa11": "spiffe://corp/reports-agent"}

def issue_credential(attestation, now=1000, ttl=300):
    """Exchange an attestation for a short-lived, workload-bound credential."""
    if not attestation:
        return None, "no attestation - unattested process"
    identity = REGISTERED.get(attestation["claims"]["image"])
    if not identity:
        return None, "image is not registered to any identity"
    return {"identity": identity, "expires": now + ttl,
            "bound_to": attestation["claims"]["node"]}, "issued"

for pid in ("proc-1", "proc-2", "proc-stolen"):
    cred, why = issue_credential(platform_attest(pid))
    print(f"   {pid:14s}{(cred['identity'] if cred else '-'):32s}{why}")

# a stolen credential presented from another node
stolen, _ = issue_credential(platform_attest("proc-1"))
def present(cred, from_node, now):
    if now > cred["expires"]:            return False, "expired"
    if from_node != cred["bound_to"]:    return False, f"bound to {cred['bound_to']}"
    return True, "accepted"

print()
for node, now in (("n-7", 1100), ("n-9", 1100), ("n-7", 2000)):
    ok, why = present(stolen, node, now)
    print(f"   presented from {node} at t={now}: {'ok' if ok else 'REFUSED'} ({why})")
print()
print("Copying the credential does not help: it is bound to a node and expires")
print("in five minutes. Copying the image does not help either - proc-2 is a")
print("real process and still gets nothing, because its image is not registered.")
assert issue_credential(platform_attest("proc-stolen"))[0] is None
assert not present(stolen, "n-9", 1100)[0]

## What you just proved

An unattested process receives no credential, a genuine but unregistered image receives none either, and a credential issued to a real workload is refused when presented from another node or after its five-minute expiry.

## Your turn

Find where one of your agents gets its first credential. If the answer is an environment variable or a mounted file, list everyone who can read it — that is the set of people who are currently that agent.

---

**Next → [A2.3 · Delegation that narrows, and survives audit](https://spbreed.github.io/cyber-commons/lessons/A2.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*